# EDA Interactive Notebook
## 🏦 Banking Exploratory Data Analysis — NexBank

---

### How to use this notebook
Run cells from top to bottom using **Shift+Enter**.
Each cell has a comment explaining what it does and why.

This notebook is the **interactive layer** on top of `run.py`.
While `run.py` produces text reports, this notebook produces:
- Visual charts (histograms, box plots, scatter plots, heatmaps)
- Interactive exploration of fraud patterns and segment behaviour
- Statistical test results with business interpretation

**Input:** `data/processed-data.csv`  
**Outputs:** `reports/analysis_report.txt` · `reports/anomalies.csv` · `reports/segment_profile.csv`

---

## Business Questions

### Question 1 — Fraud Patterns
Do fraudulent transactions cluster in specific merchant categories, channels, or time periods?  
Use `groupby` to find where fraud concentrates across the dataset.

> Answered by computing fraud rate per `merchant_category`, `channel`, and `transaction_date` (month/day).  
> A category or channel with fraud rate significantly above the dataset average is a priority for controls.  
> Time-period clustering may indicate organised fraud campaigns rather than opportunistic individual fraud.

---

### Question 2 — Customer Segment Behaviour
How do transaction amounts and frequencies differ across Retail, Premium, Business, and Student segments?  
Build a `SegmentProfiler` class that computes per-segment statistics including fraud rate.

> Each segment gets: mean amount, median amount, transaction count, total spend, and fraud rate.  
> A segment with high average transaction value **and** high fraud rate is the highest-risk cohort —  
> and the most costly to the business per fraudulent event.

---

### Question 3 — Chi-Square Independence Test
Are fraud rates statistically independent of `merchant_category`?  
Use `scipy.stats.chi2_contingency()` to test this formally.

> **H₀ (null hypothesis):** fraud occurrence is independent of merchant category  
> **H₁ (alternative):** fraud is more likely in certain merchant categories  
> If **p < 0.05**, the relationship is real and statistically significant — reject H₀.  
> If **p ≥ 0.05**, the pattern may be noise — do not over-interpret the groupby results from Q1.

---

### Question 4 — Transaction Anomalies
Flag transactions that are suspicious by amount using **IQR + Z-score consensus**.  
Also flag the inverse: transactions marked `is_fraud = True` but with a normal amount.

> Standard anomaly detection catches high-value outliers — easy fraud to spot.  
> The harder problem is low-value fraud that blends into normal spend patterns.  
> Flagging `is_fraud = True` rows with a normal amount (`z_score < 2`) isolates  
> the most evasive fraud — the type most likely to bypass automated rules engines.  
> Results saved to `reports/anomalies.csv` with an `anomaly_type` column distinguishing both groups.


In [1]:
# ── Cell 1: Imports and Setup ─────────────────────────────────────────────
# Before anything else, we import the libraries we need.
# These are the standard imports for any EDA notebook.

import sys
import pathlib
import numpy as np

# Add the project root to Python path so we can import our classes
# pathlib.Path.cwd() returns the current working directory
# We walk up until we find config.py
_root = pathlib.Path.cwd()
while not (_root / 'config.py').exists() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt   # matplotlib: the core Python plotting library
import seaborn as sns             # seaborn: statistical plots built on matplotlib
from src.eda_engine import EDAEngine, SegmentProfiler


# Tell matplotlib to display plots directly in the notebook (not in a separate window)



# Set seaborn style: 'whitegrid' adds a subtle grid which makes values easier to read
sns.set_style('whitegrid')

# Set figure size default: (width_inches, height_inches)
plt.rcParams['figure.figsize'] = (12, 5)

# Import our pipeline classes
from config import DATA_PATH, REPORTS_DIR, INDUSTRY
from src.eda_engine import EDAEngine
from src.anomaly_detector import AnomalyDetector
print(f'Setup complete. Industry: {INDUSTRY}')
print(f'Data file: {DATA_PATH}')


Setup complete. Industry: banking
Data file: /Users/samueladesina/banking-eda/data/processed-data.csv


In [ ]:
# ── Cell 2: Load and Profile the Data ────────────────────────────────────
# We use our EDAEngine class to load and profile the data.
# This is the same code that run.py executes — but here we can
# inspect intermediate results interactively.

engine = EDAEngine()
engine.load().profile()

# Access the profile results dictionary directly
p = engine.results['profile']

print('=' * 50)
print('  DATASET OVERVIEW')
print('=' * 50)
print(f'  Rows:          {p["rows"]:,}')
print(f'  Columns:       {p["columns"]}')
print(f'  Numeric cols:  {p["numeric_cols"]}')
print(f'  Text cols:     {p["categorical_cols"]}')
print(f'  Missing vals:  {p["total_nulls"]:,} ({p["null_pct"]}%)')
print(f'  Memory:        {p["memory_mb"]} MB')